In [ ]:
!pip install torch torchvision

In [ ]:
!pip install torchinfo torchmetrics tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 25.1 MB/s eta 0:00:00


In [ ]:
!pip install split-folders

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import random
import zipfile
import cv2
import splitfolders
import pandas as pd
import torch
from torch import nn
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torchinfo
from torchinfo import summary
import tqdm
from tqdm.auto import tqdm
from torchmetrics import Accuracy
from torch.optim.lr_scheduler import ReduceLROnPlateau



In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
DATA_FOLDER_PATH_ZIPPED = '/content/PlantVillage.zip'
OUTPUT_FOLDER = '/content'
WIDTH = 224
HEIGHT = 224
CHANNELS = 3


In [ ]:
DATA_FOLDERS_PATH = '/content'
with zipfile.ZipFile("/content/PlantVillage.zip","r") as zip_ref:
  print("Unzipping...")
  zip_ref.extractall(DATA_FOLDERS_PATH)

IMAGES_FOLDERS_PATH = '/content/PlantVillage'

FileNotFoundError: [Errno 2] No such file or directory: '/content/PlantVillage.zip'

In [ ]:
data = []
for class_name in os.listdir(IMAGES_FOLDERS_PATH):
    class_path = os.path.join(IMAGES_FOLDERS_PATH, class_name)
    if os.path.isdir(class_path):
        count = len(os.listdir(class_path))
        data.append({'Class': class_name, 'Count': count})

df_dist = pd.DataFrame(data).sort_values(by='Count', ascending=False)

print("Image distribution across classes:")
print(df_dist.to_string(index=False))

plt.figure(figsize=(12, 6))
plt.barh(df_dist['Class'], df_dist['Count'], color='skyblue')
plt.xlabel('Number of Images')
plt.ylabel('Class Name')
plt.title('Class Distribution (Number of images per class)')
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
def VisualizeClasses(folders_path):
    # Get all class folders and sort them
    ClassesNames = os.listdir(folders_path)
    ClassesNames.sort(key=lambda x: (x.split('__')[0], x))

    # Display the total number of classes
    NUM_CLASSES = len(ClassesNames)
    print(f"Classes Number: {NUM_CLASSES}")

    cols = 4
    rows = (NUM_CLASSES + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 3))
    axes = axes.flatten()

    # Show one sample image from each class
    for i, cat in enumerate(ClassesNames):
        ClassFolderPath = os.path.join(folders_path,cat)
        ImagesName = os.listdir(ClassFolderPath)
        img_path = os.path.join(ClassFolderPath,ImagesName[0])
        with Image.open(img_path) as img:
            width, height = img.size
        image = mpimg.imread(img_path)
        axes[i].imshow(image)
        axes[i].set_title(f"{cat}\n{width}x{height}", fontsize=10)
        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

VisualizeClasses(IMAGES_FOLDERS_PATH)

In [ ]:
# Pick a random class folder
ClassesNames = os.listdir(IMAGES_FOLDERS_PATH)
random_class = random.choice(ClassesNames)
class_path = os.path.join(IMAGES_FOLDERS_PATH, random_class)

# Pick a random image from that class
images = os.listdir(class_path)
random_image = random.choice(images)
img_path = os.path.join(class_path, random_image)

# Open and resize the selected image to 224x224
with Image.open(img_path) as img:
    resized_img = img.resize((WIDTH, HEIGHT))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(mpimg.imread(img_path))
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(resized_img)
axes[1].set_title(f"Resized ({WIDTH}x{HEIGHT})")
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def plot_healthy_vs_diseased(plant_name):
    all_classes = [d for d in os.listdir(IMAGES_FOLDERS_PATH) if d.startswith(plant_name)]

    healthy_class = [c for c in all_classes if 'healthy' in c.lower()]
    diseased_classes = [c for c in all_classes if 'healthy' not in c.lower()]

    if not healthy_class or not diseased_classes:
        print(f"Couldn't find both healthy and diseased for {plant_name}")
        return

    target_diseased = random.choice(diseased_classes)
    target_healthy = healthy_class[0]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    for i, class_name in enumerate([target_healthy, target_diseased]):
        class_path = os.path.join(IMAGES_FOLDERS_PATH, class_name)
        img_name = random.choice(os.listdir(class_path))
        img_path = os.path.join(class_path, img_name)

        img_bgr = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        axes[0, i].imshow(img_rgb)
        axes[0, i].set_title(f"Class: {class_name}")
        axes[0, i].axis('off')

        colors = ('red', 'green', 'blue')
        for j, col in enumerate(colors):
            hist = cv2.calcHist([img_rgb], [j], None, [256], [0, 256])
            axes[1, i].plot(hist, color=col, label=col)
            axes[1, i].set_xlim([0, 256])

        axes[1, i].set_title(f"Color Histogram ({'Healthy' if i==0 else 'Diseased'})")
        axes[1, i].legend()

    plt.tight_layout()
    plt.show()

plot_healthy_vs_diseased('Potato')

In [ ]:
plot_healthy_vs_diseased('Pepper')

In [ ]:
plot_healthy_vs_diseased('Tomato')

In [ ]:
plot_healthy_vs_diseased('Tomato')

In [ ]:
splitfolders.ratio(IMAGES_FOLDERS_PATH, output=OUTPUT_FOLDER, seed=42, ratio=(0.8, 0.2))

In [ ]:
TRAIN_DATA_PATH = os.path.join(OUTPUT_FOLDER, 'train')
VAL_DATA_PATH = os.path.join(OUTPUT_FOLDER, 'val')

print (f"Training data path: {TRAIN_DATA_PATH}")
print (f"Validation data path: {VAL_DATA_PATH}")

In [ ]:
WIDTH = 224
HEIGHT = 224
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

TRAIN_DATA_TRANSFORMER = transforms.Compose([
    transforms.Resize((WIDTH, HEIGHT)),
    transforms.TrivialAugmentWide(num_magnitude_bins=31),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

TEST_DATA_TRANSFORMER = transforms.Compose([
    transforms.Resize((WIDTH, HEIGHT)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

In [ ]:
train_dataset = ImageFolder(root=TRAIN_DATA_PATH, transform=TRAIN_DATA_TRANSFORMER)
val_dataset = ImageFolder(root=VAL_DATA_PATH, transform=TEST_DATA_TRANSFORMER)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")

In [ ]:
class_names = train_dataset.classes
print(f"Class names: {class_names}")

In [ ]:
class_to_idx = train_dataset.class_to_idx
print(f"Class to index mapping: {class_to_idx}")

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = os.cpu_count() if os.cpu_count() is not None else 4

TRAIN_DATA_LOADER = DataLoader(
    train_dataset,
    batch_size= BATCH_SIZE,
    shuffle=True,
    num_workers = NUM_WORKERS,
    pin_memory=True
)

TEST_DATA_LOADER = DataLoader(
    val_dataset,
    batch_size= BATCH_SIZE,
    shuffle=False,
    num_workers = NUM_WORKERS,
    pin_memory=True
)

print(f"{len(TRAIN_DATA_LOADER)} Batches in train dataloader")
print(f"{len(TEST_DATA_LOADER)} Batches in test dataloader")

In [ ]:
img,label = next(iter(TRAIN_DATA_LOADER))
img,label

In [ ]:
img.shape

In [ ]:
# Implement the CNN model
class VGG_CNN(nn.Module):
    def __init__(self, num_classes=len(class_names)):
        super().__init__()

        # Block 1: Input (3, 224, 224) -> Output (32, 112, 112)
        self.cnn_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 2: Input (32, 112, 112) -> Output (64, 56, 56)
        self.cnn_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 3: Input (64, 56, 56) -> Output (128, 28, 28)
        self.cnn_block_3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 4: Input (128, 28, 28) -> Output (256, 14, 14)
        self.cnn_block_4 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.neural_network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(14 * 14 * 256, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.cnn_block_1(x)
        x = self.cnn_block_2(x)
        x = self.cnn_block_3(x)
        x = self.cnn_block_4(x)
        x = self.neural_network(x)
        return x

base_model = VGG_CNN()

In [ ]:
summary(base_model,input_size=[1,3,224,224])

In [ ]:
loss_fn = nn.CrossEntropyLoss()
acc_fn = Accuracy(task="multiclass", num_classes=len(class_names)).to(device)
optimizer = torch.optim.Adam(params=base_model.parameters(),lr=0.001,weight_decay=1e-3)
scheduler = ReduceLROnPlateau(optimizer,
                              mode='min',
                              factor=0.1,
                              patience=3,
                              )

In [ ]:
# Create train & test step functions
def train_step(
    model: nn.Module,
    dataloader: torch.utils.data,
    loss_fn: nn.Module,
    acc_fn,
    optimizer: torch.optim,
    device=device
):
  model.train()
  total_loss, total_acc = 0,0
  for batch,(X,y) in enumerate(dataloader):
    X = X.to(device)
    y = y.to(device)
    y_logits = model(X)
    y_pred = torch.softmax(y_logits,dim=1).argmax(dim=1)
    loss = loss_fn(y_logits,y)
    acc = acc_fn(y_pred,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    total_acc += acc.item()
  total_loss /= len(dataloader)
  total_acc /= len(dataloader)
  return total_loss,total_acc


def test_step(
    model: nn.Module,
    dataloader: torch.utils.data,
    loss_fn: nn.Module,
    acc_fn,
    device=device
):
  model.eval()
  total_loss, total_acc = 0,0
  with torch.inference_mode():
    for batch,(X,y) in enumerate(dataloader):
      X = X.to(device)
      y = y.to(device)
      y_logits = model(X)
      y_pred = torch.softmax(y_logits,dim=1).argmax(dim=1)
      total_loss += loss_fn(y_logits,y).item()
      total_acc += acc_fn(y_pred,y).item()
    total_loss /= len(dataloader)
    total_acc /= len(dataloader)
  return total_loss,total_acc


In [ ]:
# Create main train loop for epochs
SAVE_FROM_SCRATCH_MODEL_PATH = f"../model_from_scratch.pth"
def train(model: nn.Module,
          train_dataloader: torch.utils.data,
          test_dataloader: torch.utils .data,
          loss_fn: nn.Module,
          acc_fn,
          optimizer: torch.optim,
          epochs: int,
          patience: int = 5,
          seed=42,
          ):
  torch.random.manual_seed(seed)
  results = {
      'Train Loss': [],
      'Train Accuracy': [],
      'Test Loss': [],
      'Test Accuracy': []
  }
  best_loss = float('inf')
  patience_counter = 0
  for epoch in tqdm(range(epochs)):
    train_loss,train_acc = train_step(model,train_dataloader,loss_fn,acc_fn,optimizer)
    test_loss,test_acc = test_step(model,test_dataloader,loss_fn,acc_fn)
    scheduler.step(test_loss)
    if test_loss < best_loss:
        best_loss = test_loss
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_FROM_SCRATCH_MODEL_PATH)
    else:
        patience_counter += 1
    print(f"Epoch {epoch+1}/{epochs} - accuracy: {train_acc*100:.2f}% - loss: {train_loss:.4f} - val_accuracy: {test_acc*100:.2f}% - val_loss: {test_loss:.4f}")
    results["Train Loss"].append(train_loss),results["Train Accuracy"].append(train_acc),results["Test Loss"].append(test_loss),results["Test Accuracy"].append(test_acc)
    if patience_counter >= patience:
            print(f"--- Early stopping triggered at epoch {epoch+1} ---")
            break
  model.load_state_dict(torch.load(SAVE_FROM_SCRATCH_MODEL_PATH))
  return results



In [ ]:
base_model_results = train(base_model,TRAIN_DATA_LOADER,TEST_DATA_LOADER,loss_fn,acc_fn,optimizer,30)

In [ ]:
def get_predections(model: nn.Module, dataloader: torch.utils.data, device=device):
  model_prediction = []
  model.eval()
  with torch.inference_mode():
    for batch,(X,y) in tqdm(enumerate(dataloader)):
      X = X.to(device)
      y = y.to(device)
      y_logits = model(X)
      y_preds = torch.softmax(y_logits,dim=1).argmax(dim=1)
      model_prediction.append(y_preds)
  return torch.cat(model_prediction)

model0_predictions = get_predections(base_model,TEST_DATA_LOADER)
model0_predictions


In [ ]:
base_model_results_table = pd.DataFrame()
base_model_results_table